In [1]:
!pip install -q -U \
    transformers \
    peft \
    trl \
    datasets \
    accelerate \
    bitsandbytes

import transformers, peft, trl, datasets, accelerate, bitsandbytes
print(f"transformers: {transformers.__version__}")
print(f"peft:         {peft.__version__}")
print(f"trl:          {trl.__version__}")
print(f"datasets:     {datasets.__version__}")
print(f"accelerate:   {accelerate.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print("\n✅ All installed! RESTART runtime now.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.3/616.3 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 84.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 23

In [1]:
import torch
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

import transformers, peft, trl, bitsandbytes
print(f"transformers: {transformers.__version__}")
print(f"peft:         {peft.__version__}")
print(f"trl:          {trl.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print("\n✅ GPU ready!")

GPU : Tesla T4
VRAM: 14.6 GB
CUDA: 12.6
transformers: 5.3.0
peft:         0.18.1
trl:          0.29.0
bitsandbytes: 0.49.2

✅ GPU ready!


In [4]:
from huggingface_hub import login
login()  # Paste your HF token (needs access to meta-llama/Llama-3.2-3B)

In [5]:
import os
import shutil

UPLOAD_DIR = "/content"
DATA_DIR = "/content/data/processed"
os.makedirs(DATA_DIR, exist_ok=True)

expected_files = [
    "summarization_prepared.jsonl",
    "persona_prepared.jsonl",
    "chat_prepared.jsonl",
]

found = []
for f in expected_files:
    src = os.path.join(UPLOAD_DIR, f)
    dst = os.path.join(DATA_DIR, f)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        lines = sum(1 for _ in open(dst, 'r', encoding='utf-8'))
        size_mb = os.path.getsize(dst) / 1024**2
        print(f"  ✅ {f}: {lines:,} samples ({size_mb:.1f} MB)")
        found.append(f)
    elif os.path.exists(dst):
        lines = sum(1 for _ in open(dst, 'r', encoding='utf-8'))
        size_mb = os.path.getsize(dst) / 1024**2
        print(f"  ✅ {f}: {lines:,} samples ({size_mb:.1f} MB) [already in place]")
        found.append(f)
    else:
        print(f"  ⏭️  {f}: not found (will be skipped)")

if not found:
    print("\n⚠️  No datasets found! Upload files first.")
else:
    print(f"\n✅ {len(found)} dataset(s) ready!")

  ✅ summarization_prepared.jsonl: 603 samples (19.0 MB)
  ✅ persona_prepared.jsonl: 241 samples (0.0 MB)
  ⏭️  chat_prepared.jsonl: not found (will be skipped)

✅ 2 dataset(s) ready!


In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "meta-llama/Llama-3.2-3B"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype="auto",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"\n✅ Model loaded: {MODEL_ID}")
print(f"   Vocab size: {len(tokenizer):,}")
print(f"   Model dtype: 8-bit quantized")

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]


✅ Model loaded: meta-llama/Llama-3.2-3B
   Vocab size: 128,256
   Model dtype: 8-bit quantized


In [7]:
# Free ~6 GB by clearing the HuggingFace download cache (model is already loaded in memory)
!rm -rf /root/.cache/huggingface/hub/models--meta-llama--Llama-3.2-3B
!rm -rf /content/=0.43.0
!df -h /content | tail -1 | awk '{print "💾 Disk free: " $4}'


💾 Disk free: 25G


In [8]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=6,
    lora_alpha=8,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
print("\n✅ LoRA adapters attached (base model frozen)")

trainable params: 9,117,696 || all params: 3,221,867,520 || trainable%: 0.2830

✅ LoRA adapters attached (base model frozen)


In [10]:
import json, os

DATA_DIR = "/content/data/processed"
for fname in ["summarization_prepared.jsonl", "persona_prepared.jsonl"]:
    fpath = os.path.join(DATA_DIR, fname)
    if not os.path.exists(fpath):
        continue
    bad = 0
    total = 0
    with open(fpath, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            total += 1
            try:
                json.loads(line)
            except json.JSONDecodeError:
                bad += 1
                if bad <= 3:
                    print(f"  ❌ {fname} line {i}: {line[:80]}...")
    print(f"  {fname}: {total} lines, {bad} bad\n")


  ❌ summarization_prepared.jsonl line 603: {"instruction": "Summarize the following scientific document:", "input": "n49b i...
  summarization_prepared.jsonl: 603 lines, 1 bad

  persona_prepared.jsonl: 241 lines, 0 bad



In [11]:
import json, os

DATA_DIR = "/content/data/processed"
dataset_files = [
    "summarization_prepared.jsonl",
    "persona_prepared.jsonl",
    "chat_prepared.jsonl",
]

all_records = []
skipped = 0
for fname in dataset_files:
    fpath = os.path.join(DATA_DIR, fname)
    if not os.path.exists(fpath):
        print(f"  ⏭️  Skipped (not found): {fname}")
        continue
    count = 0
    with open(fpath, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                all_records.append(json.loads(line))
                count += 1
            except json.JSONDecodeError:
                skipped += 1
    print(f"  ✅ Loaded {count:,} samples from {fname}")

print(f"\n📊 Total merged: {len(all_records):,} samples ({skipped} skipped)")

from datasets import Dataset
merged_dataset = Dataset.from_list(all_records)
print(f"   Columns: {merged_dataset.column_names}")


  ✅ Loaded 602 samples from summarization_prepared.jsonl
  ✅ Loaded 241 samples from persona_prepared.jsonl
  ⏭️  Skipped (not found): chat_prepared.jsonl

📊 Total merged: 843 samples (1 skipped)
   Columns: ['instruction', 'input', 'output']


In [12]:
def format_instruction(example):
    instruction = example["instruction"].strip()
    input_text = example["input"].strip()
    output_text = example["output"].strip()

    if input_text:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n{output_text}"
        )
    else:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{output_text}"
        )
    return {"text": text}

formatted_dataset = merged_dataset.map(
    format_instruction,
    remove_columns=merged_dataset.column_names,
    desc="Formatting",
)

split = formatted_dataset.train_test_split(test_size=0.1, seed=42)
dataset_dict = DatasetDict({
    "train": split["train"],
    "validation": split["test"],
})

print(f"✅ Train: {len(dataset_dict['train']):,} | Val: {len(dataset_dict['validation']):,}")
print(f"\n📋 Sample:")
print(dataset_dict["train"][0]["text"][:400])

Formatting:   0%|          | 0/843 [00:00<?, ? examples/s]

✅ Train: 758 | Val: 85

📋 Sample:
### Instruction:
Summarize the following scientific document:

### Input:
in the landmark papers by foschini and gans @xcite and by telatar @xcite the great promise of the use of multiple transmit and receive antennas was presented and established . the importance of such multiple input multiple output ( mimo ) links is based on the fact that parallel data streams emanating from different transmit


In [13]:
from transformers import TrainerCallback

SAMPLE_PROMPTS = [
    {
        "task": "Summarization",
        "text": (
            "### Instruction:\n"
            "Summarize the following scientific document:\n\n"
            "### Input:\n"
            "Deep learning has achieved remarkable success in NLP, "
            "computer vision, and speech recognition. Recent advances "
            "in transformer architectures have enabled models to capture "
            "long-range dependencies more effectively.\n\n"
            "### Response:\n"
        ),
    },
    {
        "task": "Persona (Chandler)",
        "text": (
            "### Instruction:\n"
            "Reply as Chandler Bing from Friends:\n\n"
            "### Input:\n"
            "Do you want to go to the gym with me?\n\n"
            "### Response:\n"
        ),
    },
]


class SampleGenerationCallback(TrainerCallback):
    def __init__(self, every_n_steps=200):
        self.every_n_steps = every_n_steps

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.every_n_steps != 0 or state.global_step == 0:
            return
        mdl = kwargs.get("model", None)
        if mdl is None:
            return
        mdl.eval()
        print(f"\n{'='*60}")
        print(f"  📝 Sample Generations @ Step {state.global_step}")
        print(f"{'='*60}")
        for sample in SAMPLE_PROMPTS:
            inputs = tokenizer(sample["text"], return_tensors="pt", truncation=True, max_length=512).to(mdl.device)
            with torch.no_grad():
                outputs = mdl.generate(**inputs, max_new_tokens=150, do_sample=True, temperature=0.7, top_p=0.9, repetition_penalty=1.1)
            generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
            response = generated[len(sample["text"]):].strip()
            print(f"\n  [{sample['task']}]")
            print(f"  → {response[:300]}")
        print(f"{'='*60}\n")
        mdl.train()


sample_callback = SampleGenerationCallback(every_n_steps=200)
print("✅ Callback ready (generates samples every 200 steps)")

✅ Callback ready (generates samples every 200 steps)


In [17]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./out",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=False,                # Disabled — 8-bit quantization handles precision
    logging_steps=100,
    save_steps=500,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=200,
    max_grad_norm=0.3,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["validation"],
    processing_class=tokenizer,
    callbacks=[sample_callback],
)

model.config.use_cache = False
print("🚀 Ready to train!")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/758 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/758 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/758 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/85 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/85 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/85 [00:00<?, ? examples/s]

🚀 Ready to train!


In [18]:
trainer.train()
print("\n✅ Training complete!")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss,Validation Loss
200,2.154594,2.219364



  📝 Sample Generations @ Step 200

  [Summarization]
  → This article explores deep learning approaches for natural language processing (NLP) tasks such as machine translation, summarization, and question answering. The authors argue that these techniques are particularly powerful when dealing with complex linguistic phenomena like semantics, context, and

  [Persona (Chandler)]
  → I have a better idea. Let's go get a massage.

### Response:
You know, I can't tell if that was an answer or not...

### Response:
That wasn't an answer either.

### Response:
Maybe we should do both!

### Response:
It'll be like a mini vacation!



Step,Training Loss,Validation Loss
200,2.154594,2.219364


config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]


✅ Training complete!


In [19]:
OUTPUT_DIR = "/content/out"

trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Verify key files
import os
for name in ["adapter_model.safetensors", "adapter_model.bin", "adapter_config.json"]:
    path = os.path.join(OUTPUT_DIR, name)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024**2
        print(f"  ✅ {name} ({size_mb:.2f} MB)")

print(f"\n✅ Saved to {OUTPUT_DIR}/")

  ✅ adapter_model.safetensors (17.44 MB)
  ✅ adapter_config.json (0.00 MB)

✅ Saved to /content/out/


In [20]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy adapter to Drive
!cp -r /content/out /content/drive/MyDrive/llama3_lora_phase1
!ls -lh /content/drive/MyDrive/llama3_lora_phase1/
print("✅ Saved to Google Drive! Safe even if runtime disconnects.")


Mounted at /content/drive
total 34M
-rw------- 1 root root 1.1K Mar 19 09:03 adapter_config.json
-rw------- 1 root root  18M Mar 19 09:03 adapter_model.safetensors
drwx------ 2 root root 4.0K Mar 19 09:03 checkpoint-285
-rw------- 1 root root 1.5K Mar 19 09:03 README.md
-rw------- 1 root root  335 Mar 19 09:03 tokenizer_config.json
-rw------- 1 root root  17M Mar 19 09:03 tokenizer.json
✅ Saved to Google Drive! Safe even if runtime disconnects.


In [21]:
metrics = trainer.state.log_history
train_losses = [(m["step"], m["loss"]) for m in metrics if "loss" in m and "eval_loss" not in m]
eval_losses = [(m["step"], m["eval_loss"]) for m in metrics if "eval_loss" in m]

print("📊 Training Log:")
print(f"   Final train loss: {train_losses[-1][1]:.4f}" if train_losses else "   No train losses")
print(f"   Final eval loss:  {eval_losses[-1][1]:.4f}" if eval_losses else "   No eval losses")
print(f"   Total steps: {trainer.state.global_step}")
print(f"   Epochs: {trainer.state.epoch:.1f}")

📊 Training Log:
   Final train loss: 2.1546
   Final eval loss:  2.2194
   Total steps: 285
   Epochs: 3.0


In [22]:
model.eval()
test_prompts = [
    "### Instruction:\nSummarize the following conversation:\n\n### Input:\nAlice: Hey, are we meeting at 3?\nBob: Yes! Same place.\nAlice: Perfect.\n\n### Response:\n",
    "### Instruction:\nReply as Chandler Bing from Friends:\n\n### Input:\nI just got a promotion!\n\n### Response:\n",
]

print("🧪 Quick Inference Test:\n")
for i, prompt in enumerate(test_prompts, 1):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7, top_p=0.9)
    response = tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):].strip()
    print(f"Test {i}: {response[:200]}\n")

🧪 Quick Inference Test:



/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Test 1: Alice asks Bob whether they are meeting at 3 o'clock. Bob confirms that they are meeting at that time.

Test 2: Congratulations! You deserve it!



In [23]:
!cd /content && zip -r /content/llama3_lora_phase1.zip out/
!ls -lh /content/llama3_lora_phase1.zip
print("\n✅ Ready for download!")

  adding: out/ (stored 0%)
  adding: out/tokenizer.json (deflated 85%)
  adding: out/adapter_config.json (deflated 58%)
  adding: out/checkpoint-285/ (stored 0%)
  adding: out/checkpoint-285/rng_state.pth (deflated 26%)
  adding: out/checkpoint-285/tokenizer.json (deflated 85%)
  adding: out/checkpoint-285/training_args.bin (deflated 53%)
  adding: out/checkpoint-285/adapter_config.json (deflated 58%)
  adding: out/checkpoint-285/optimizer.pt (deflated 10%)
  adding: out/checkpoint-285/trainer_state.json (deflated 61%)
  adding: out/checkpoint-285/adapter_model.safetensors (deflated 22%)
  adding: out/checkpoint-285/README.md (deflated 65%)
  adding: out/checkpoint-285/scheduler.pt (deflated 61%)
  adding: out/checkpoint-285/tokenizer_config.json (deflated 45%)
  adding: out/adapter_model.safetensors (deflated 22%)
  adding: out/README.md (deflated 43%)
  adding: out/tokenizer_config.json (deflated 45%)
-rw-r--r-- 1 root root 49M Mar 19 09:08 /content/llama3_lora_phase1.zip

✅ Ready fo

In [24]:
try:
    from google.colab import files
    files.download('/content/llama3_lora_phase1.zip')
    print('📥 Download started!')
except ImportError:
    print('Download /content/llama3_lora_phase1.zip from Files panel.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Download started!
